## Overview

In this notebook, we will utilize the AIEnrichment Service to execute the Azure OpenAI model, transforming it into the standard AIEnrichment Output schema. This process includes several steps: configuration management, model setup, input preparation, and execution of the enrichment service.

#### Prerequisites
- **Azure OpenAI Model Deployment:** Ensure you have a deployed Azure OpenAI model. Refer to the Model Deployment in Azure OpenAI documentation [here](https://learn.microsoft.com/en-us/azure/ai-services/openai/overview).

- **DAX data Integration capability:** Ensure you have the DAX Copilot Integration capability set up.  Refer to the DAX data Integration capability documentation [here](https://learn.microsoft.com/en-us/industry/healthcare/dax-copilot-integration/overview).

####  Input Preparation
- Define the Enrichment Definition to prepare the enrichment input data.
- Specify metadata and text file references for the enrichment process.

####  AI Enrichment Service Execution
- Execute the enrichment service using the Enrichment ID.

####  Configurations
Before running this notebook, ensure that all configurations in the msft_config_notebook are completed.

###### Configuration management and setup

To setup and manage configurations for the Healthcare data solutions, please execute the following cell:

In [ ]:
%run msft_config_notebook

In [ ]:
%run msft_config_notebook {"enable_spark_setup" : true, "enable_packages_mount" : false}

In [ ]:
from microsoft.fabric.hls.hds.ai_enrichments.use_cases import ConversationalDataTransformer,ConversationalDataModelProcessor
from microsoft.fabric.hls.hds.ai_enrichments.core.services.ai_enrichments_service import AIEnrichmentsService
from microsoft.fabric.hls.hds.ai_enrichments.core import EnrichmentView,Enrichment,EnrichmentViewExpression,EnrichmentDefinition,EnrichmentInputMapping,EnrichmentFileReference,EnrichmentViewDefinition

##### Configuration for Conversational Data Enrichment

In [ ]:
# Inline Parameters
INLINES_PARAMS={ 
        'refresh-metadata':True # Set to TRUE on the initial run or whenever new tables are added to silver
}

##### Initialize Processor and Transformer

In [ ]:
conversational_data_processor = ConversationalDataModelProcessor()

conversational_data_transformer=ConversationalDataTransformer()

##### Initialize AIEnrichment Service

In [ ]:
#AI Enrichment Service
ai_enrichments_service=AIEnrichmentsService(
        spark,
        workspace_name=workspace_name,
        solution_name=solution_name,
        admin_lakehouse_name=administration_database_name,
        inline_params=INLINES_PARAMS,
        enrichment_model_processor=conversational_data_processor,
        enrichment_transformer=conversational_data_transformer,
        one_lake_endpoint=one_lake_endpoint)



### AIEnrichment Definition

#### Create View Definition



In [ ]:
# METADATA DEFINITION VARIABLES
ENRICHMENT_VIEW_TABLES = ["DaxTranscripts"]

SQL_QUERY="SELECT transcriptId,content,encounterId,patientId FROM view1"

PARENT_VIEW_IDS = ai_enrichments_service.metadata.get_enrichment_view_ids(ENRICHMENT_VIEW_TABLES)

sql_expression=EnrichmentViewExpression(

    type="sql",
    query=SQL_QUERY
)

enrichment_view_definition=EnrichmentViewDefinition(
    parent_views_ids=PARENT_VIEW_IDS,
    expression=sql_expression
)

  # Create an instance of EnrichmentView  
enrichment_view = EnrichmentView(      
    name='View for Conversational Data',  
    description='View for extracting enrichments for Conversational Data',  
    definition=enrichment_view_definition
)  

enrichment_view_id=ai_enrichments_service.metadata.create_enrichment_view(enrichment_view)

#### Create Enrichment Definition

In [ ]:
# Define OpenAI API settings 
OPENAI_API_KEY_SECRET_NAME = ""  
OPENAI_API_ENDPOINT = ""  
OPENAI_API_VERSION="2024-08-01-preview"
OPENAI_MODEL_NAME = "gpt-4o"

model_definition = {  
    "api_key_secret_name": OPENAI_API_KEY_SECRET_NAME,  
    "api_endpoint": OPENAI_API_ENDPOINT,  
    "version": OPENAI_API_VERSION,  
    "name": OPENAI_MODEL_NAME
}  

In [ ]:

metadata_dict={}
  
column_references=EnrichmentFileReference(  id="transcriptId",   content="content"   ) 

input_mapping=EnrichmentInputMapping(  
            patient_id="patientId",  
            metadata=metadata_dict,  
            text_resource_references=[column_references]  
        )


# Create an instance of Enrichment  
enrichment = Enrichment(  
    name='Enrichment for Conversational Data',  
    description='Definition for Conversational Data Enrichment',  
    definition=EnrichmentDefinition(  
        model=model_definition,  
        view_id=f"{enrichment_view_id}",  
        input_mapping=input_mapping
    )  
)    

enrichment_id=ai_enrichments_service.metadata.create_enrichment(enrichment)

### AIEnrichment Execution

In [ ]:
ai_enrichments_service.execution.execute(enrichment_id)